
# 📚 Análise de Base de Livros com SQL

Este projeto utiliza **SQL e PostgreSQL** para explorar uma base relacional de livros, autores, editoras, classificações e resenhas.

O objetivo é identificar padrões do catálogo e do comportamento dos usuários que possam apoiar decisões para um produto digital voltado a leitores.

## Objetivos da análise

- identificar quantos livros foram publicados após 1º de janeiro de 2000;
- calcular o número de resenhas e a classificação média de cada livro;
- identificar a editora com mais livros de tamanho relevante;
- encontrar o autor com maior média de classificação entre livros com volume mínimo de avaliações;
- analisar o engajamento dos usuários mais ativos.



## 1. Modelo de dados

A análise utiliza cinco tabelas:

| Tabela | Conteúdo principal | Chaves relevantes |
|---|---|---|
| `books` | livros, data de publicação, páginas, autor e editora | `book_id`, `author_id`, `publisher_id` |
| `authors` | nomes dos autores | `author_id` |
| `publishers` | nomes das editoras | `publisher_id` |
| `ratings` | classificações numéricas dadas pelos usuários | `rating_id`, `book_id`, `username` |
| `reviews` | resenhas textuais escritas pelos usuários | `review_id`, `book_id`, `username` |

As relações principais são feitas por `book_id`, `author_id` e `publisher_id`.



## 2. Conexão com o PostgreSQL

> **Segurança:** credenciais de banco não são armazenadas neste notebook.  
> Para executar o projeto, configure as variáveis de ambiente indicadas abaixo com credenciais válidas.


In [ ]:

import os
import pandas as pd
from sqlalchemy import create_engine

# Credenciais devem ser definidas como variáveis de ambiente.
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME')

if not all([DB_USER, DB_PASSWORD, DB_HOST, DB_NAME]):
    raise ValueError(
        'Configure DB_USER, DB_PASSWORD, DB_HOST e DB_NAME antes de executar a conexão.'
    )

connection_string = (
    f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

engine = create_engine(
    connection_string,
    connect_args={'sslmode': 'require'}
)



## 3. Inspeção inicial das tabelas

Antes das consultas analíticas, as primeiras linhas das cinco tabelas são verificadas para compreender sua estrutura.


In [ ]:

tables = ['books', 'authors', 'publishers', 'ratings', 'reviews']

for table in tables:
    print(f'\n--- {table.upper()} ---')
    display(pd.read_sql(f'SELECT * FROM {table} LIMIT 5;', con=engine))



## 4. Livros publicados após 1º de janeiro de 2000

A primeira consulta mede o volume de publicações mais recentes no catálogo.


In [ ]:

query_1 = '''
SELECT COUNT(*) AS total_livros
FROM books
WHERE publication_date > '2000-01-01';
'''

result_1 = pd.read_sql(query_1, con=engine)
result_1



### Resultado

Foram identificados **819 livros publicados após 1º de janeiro de 2000**.

Esse resultado mostra que uma parcela relevante do catálogo é composta por publicações posteriores ao início dos anos 2000.



## 5. Número de resenhas e classificação média por livro

As tabelas `reviews` e `ratings` representam métricas diferentes:

- `reviews`: resenhas textuais;
- `ratings`: notas/classificações numéricas.

Para evitar multiplicação indevida de linhas ao combinar as duas tabelas, cada métrica é agregada separadamente antes dos `JOINs`.


In [ ]:

query_2 = '''
SELECT
    b.book_id,
    b.title,
    COALESCE(rv.num_reviews, 0) AS num_reviews,
    ROUND(rt.avg_rating::numeric, 2) AS avg_rating
FROM books AS b
LEFT JOIN (
    SELECT
        book_id,
        COUNT(review_id) AS num_reviews
    FROM reviews
    GROUP BY book_id
) AS rv
    ON b.book_id = rv.book_id
LEFT JOIN (
    SELECT
        book_id,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
) AS rt
    ON b.book_id = rt.book_id
ORDER BY b.book_id;
'''

result_2 = pd.read_sql(query_2, con=engine)
result_2



### Interpretação

A consulta mantém todos os livros da tabela `books`, inclusive aqueles que eventualmente não possuam resenhas, e separa corretamente:

- **quantidade de resenhas textuais**;
- **média das classificações numéricas**.

Essa distinção torna a análise de engajamento mais precisa.



## 6. Editora com maior número de livros com mais de 50 páginas

Publicações muito curtas são excluídas para concentrar a análise em livros com conteúdo mais substancial.


In [ ]:

query_3 = '''
SELECT
    p.publisher,
    COUNT(b.book_id) AS num_books
FROM books AS b
JOIN publishers AS p
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher_id, p.publisher
ORDER BY num_books DESC
LIMIT 1;
'''

result_3 = pd.read_sql(query_3, con=engine)
result_3



### Resultado

A **Penguin Books** apresentou a maior quantidade de livros com mais de 50 páginas, com **42 livros**.

Esse indicador ajuda a identificar editoras com forte presença no catálogo analisado.



## 7. Autor com maior média de classificação

Nesta etapa são considerados apenas livros com pelo menos **50 classificações**, reduzindo o peso de obras avaliadas por poucos usuários.


In [ ]:

query_4 = '''
SELECT
    a.author,
    ROUND(AVG(r.rating)::numeric, 2) AS avg_rating
FROM authors AS a
JOIN books AS b
    ON a.author_id = b.author_id
JOIN ratings AS r
    ON b.book_id = r.book_id
WHERE b.book_id IN (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)
GROUP BY a.author_id, a.author
ORDER BY avg_rating DESC
LIMIT 1;
'''

result_4 = pd.read_sql(query_4, con=engine)
result_4



### Resultado

**J.K. Rowling/Mary GrandPré** apresentou a maior média de classificação, aproximadamente **4,29**, considerando apenas livros com pelo menos 50 classificações.

O filtro mínimo de avaliações torna o resultado mais representativo.



## 8. Engajamento dos usuários mais ativos

Primeiro são identificados os usuários que classificaram mais de **50 livros**. Em seguida, calcula-se quantas **resenhas textuais** esses mesmos usuários produziram.


In [ ]:

query_5 = '''
WITH active_users AS (
    SELECT
        username
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),
reviews_by_user AS (
    SELECT
        au.username,
        COUNT(r.review_id) AS num_reviews
    FROM active_users AS au
    LEFT JOIN reviews AS r
        ON au.username = r.username
    GROUP BY au.username
)
SELECT
    ROUND(AVG(num_reviews)::numeric, 2) AS avg_reviews
FROM reviews_by_user;
'''

result_5 = pd.read_sql(query_5, con=engine)
result_5



### Interpretação

Na versão inicial do projeto, o cálculo foi feito sobre as linhas de `ratings`, resultando em **54,33 classificações por usuário ativo**.

Para a versão de portfólio, a consulta acima diferencia corretamente **classificações** de **resenhas textuais**. A média de resenhas desses usuários fica em torno de **24 por usuário**, evidenciando que classificar um livro é uma ação mais frequente do que escrever uma resenha completa.



# 9. Conclusão geral

A análise SQL permitiu explorar aspectos importantes do catálogo e do engajamento dos usuários:

- **819 livros** foram publicados após 1º de janeiro de 2000;
- **Penguin Books** liderou entre as editoras, com **42 livros** de mais de 50 páginas;
- **J.K. Rowling/Mary GrandPré** apresentou média aproximada de **4,29** entre autores com livros suficientemente avaliados;
- a separação entre `ratings` e `reviews` mostrou que notas e resenhas representam níveis diferentes de engajamento;
- usuários que classificam muitos livros constituem um segmento relevante para recursos de recomendação, rankings e comunidade.

## Recomendações

Os resultados podem apoiar funcionalidades como:

- recomendações personalizadas;
- rankings de livros por classificação;
- destaques de autores e editoras;
- filtros por popularidade e avaliação;
- recursos voltados a usuários altamente engajados;
- incentivo à produção de resenhas textuais para enriquecer a experiência da comunidade.



## Tecnologias utilizadas

- SQL
- PostgreSQL
- Python
- Pandas
- SQLAlchemy
- Jupyter Notebook
